In [124]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import requests
import concurrent.futures
import time

In [125]:
def adjust_rank(href,page):
    try:
        rank = int(href.split("ref=sr_1_")[1].split("?")[0])
        if page == 1:
            if rank<=4 or (rank >= 21 and rank <= 24):
                return 0
            
            if rank <= 20:
                rank -= 4
            elif rank > 24 and rank <= 56:
                rank -= 8

        if page == 2:
            if (rank >= 49 and rank <= 52) or (rank >= 69 and rank <= 72):
                return 0
            
            if rank <= 68:
                rank -= 4
            elif rank > 72 and rank <= 104:
                rank -= 8

    except:
        rank = href.split("ref=sr_1_")[1].split("?")[0]

    return rank

In [126]:
def scrape_item(item_data):
    output = []
    driver = webdriver.Remote(command_executor="http://localhost:4444" , options=options)
    # driver = webdriver.Chrome(service=service , options=options)
    for pincode in pincodes:
        asin_found = False
        page = 1
        while(not asin_found):
            
            if page==1:
                try:
                    wait = WebDriverWait(driver, 3)
                    driver.get("https://www.amazon.in/")
                    wait.until(EC.presence_of_element_located((By.ID,"nav-global-location-popover-link"))).click()
                    location_input = wait.until(EC.presence_of_element_located((By.ID,"GLUXZipUpdateInput")))
                    location_input.clear()
                    location_input.send_keys(pincode)
                    location_input.send_keys(Keys.RETURN)

                    
                    # location_ele = driver.find_element(By.ID, 'nav-global-location-popover-link')
                    # location_ele.click()
                    # # time.sleep(1)
                    # # location_input = driver.find_element(By.ID, 'GLUXZipUpdateInput') 
                    # location_input = wait.until(EC.presence_of_element_located((By.ID,"GLUXZipUpdateInput")))
                    # location_input.clear()
                    # location_input.send_keys(pincode)
                    # driver.find_element(By.ID, 'GLUXZipUpdate').click()

                    
                    time.sleep(1)
                except Exception as e:
                    print(e)
                    output.append({
                        "pincode":pincode,
                        "keyword":item_data["keyword"],
                        "asin":item_data["asin"],
                        "rank":"---"
                    })
                    break
            driver.get("https://www.amazon.in/s?k=" + item_data["keyword"] + "&page=" + str(page))
            divs = driver.find_elements(By.CSS_SELECTOR, f'div[data-asin="{item_data["asin"]}"]')
            if len(divs) == 0:
                page+=1
                if(page > max_pages):
                    # print(f"item {item_data["asin"]} not found in {max_pages} pages on pin {pincode}")
                    output.append({
                        "pincode":pincode,
                        "keyword":item_data["keyword"],
                        "asin":item_data["asin"],
                        "rank":">96"
                    })
                    break
                for div in divs:
                    try:
                        anchor = div.find_elements(By.TAG_NAME, "a")
                        href = anchor[0].get_attribute("href")
                        if "ref=sr_1_" in href:
                            rank = adjust_rank(href,page)
                            if rank == 0:
                                break
            
                            output.append({
                                "pincode":pincode,
                                "keyword":item_data["keyword"],
                                "asin":item_data["asin"],
                                "rank":rank
                            })
                            found_asin = True
                            # print(f"item {item_data["asin"]} found on pin {pincode}")
                    except:
                        print("anchor error")
                        page += 1
                        if(page > max_pages):
                            # print(f"item {item_data["asin"]} not found in {max_pages} pages on pin {pincode}")
                            output.append({
                                "pincode":pincode,
                                "keyword":item_data["keyword"],
                                "asin":item_data["asin"],
                                "rank":">96"
                            })
                            break
    driver.quit()
    print(str(item_data["n"])+",",end="")
    return output
        

In [127]:
options = webdriver.ChromeOptions()
options.add_argument('--headless=new')
options.add_argument("--incognito")
pincodes = [
    "110001",
    "700001",
    "400008",
    "560056"
]
max_pages = 2
data_in = eval(requests.get("https://script.google.com/macros/s/AKfycbxugMsk0xHYo1-Fz3HIkgc9b6M9OtvjNjyMbymKVctiz1Cw1w7S6xmgSr497AXP8o0j/exec?sheet=EB_DATA").text)

final = []
n = 0
for data in data_in[1:]:
    if ((len(str(data[3])) > 1) and (data[4]!= "#VALUE!")):
        keyw = 0
        for keyword in data[4:]:
            final.append({
                "n":n,
                "asin":data[0],
                "keyword":keyword
            })
            n+=1
            keyw += 1
            if keyw == 28:
                break

final = final[:28]
start=time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(scrape_item,final))
out = [item for items in results for item in items]

r = requests.post("https://script.google.com/macros/s/AKfycbxugMsk0xHYo1-Fz3HIkgc9b6M9OtvjNjyMbymKVctiz1Cw1w7S6xmgSr497AXP8o0j/exec?type=ranking&sheet=EB_RANK",json=out)
print(r.text)
print(f"Done in {time.time()-start}")

Message: 
Stacktrace:
0   chromedriver                        0x00000001027faa80 chromedriver + 4385408
1   chromedriver                        0x00000001027f338c chromedriver + 4354956
2   chromedriver                        0x0000000102410b0c chromedriver + 281356
3   chromedriver                        0x00000001024532f8 chromedriver + 553720
4   chromedriver                        0x000000010248bd24 chromedriver + 785700
5   chromedriver                        0x0000000102447eec chromedriver + 507628
6   chromedriver                        0x00000001024488c4 chromedriver + 510148
7   chromedriver                        0x00000001027c243c chromedriver + 4154428
8   chromedriver                        0x00000001027c6ea0 chromedriver + 4173472
9   chromedriver                        0x00000001027a7ff8 chromedriver + 4046840
10  chromedriver                        0x00000001027c778c chromedriver + 4175756
11  chromedriver                        0x000000010279afb8 chromedriver + 3993528